# 06 - Retrieval


## What this notebook is for

This notebook is written as **notes + code**. Read the Markdown cells first, then run the code cells.

The goal is to understand the concept, see the architecture, implement a small example, and then evaluate/improve it.

## Learning objectives

- Understand semantic, keyword, and hybrid retrieval.
- Learn top-k, similarity thresholds, metadata filtering, and MMR.
- Understand why retrieval quality controls RAG quality.
- Create a simple retrieval evaluation set.

## Core AI concept

A retriever turns a user query into a ranked set of candidate documents. Retrieval is a ranking problem. The retriever should maximize useful evidence while minimizing irrelevant context.

## Architecture / mental model

```text
User query
   ↓
Query transformation
   ↓
Retriever
   ├── semantic
   ├── keyword
   └── hybrid
   ↓
rank/filter
   ↓
context
```

## Connection to the supplied learning material

Your `searching.ipynb` compares FAISS L2 search, cosine similarity, and Euclidean distance. The new notebook connects those low-level ideas to LangChain retriever behavior.

### Source foundation
This notebook is an expanded learning version of the uploaded notebooks. The source material for this track includes:
- `chunking.ipynb`
- `embeddings.ipynb`
- `faiss.ipynb`
- `chroma_db.ipynb`
- `searching.ipynb`
- `rag.ipynb`

The original examples and terminology are retained where useful, then organized into a cleaner learning sequence.

## Setup

The code below uses environment variables for secrets and model names so the notebook is easier to maintain when providers or model versions change.

In [ ]:
from pathlib import Path
import os
from dotenv import load_dotenv

load_dotenv()

OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-5.6-luna")
OPENAI_EMBED_MODEL = os.getenv("OPENAI_EMBED_MODEL", "text-embedding-3-small")

print("Model:", OPENAI_MODEL)
print("Embedding model:", OPENAI_EMBED_MODEL)

In [ ]:
%pip install -qU langchain-chroma langchain-openai langchain-core

### Basic retriever

In [ ]:
from langchain_core.documents import Document

docs = [
    Document(page_content="Python is used for data science.", metadata={"topic": "python"}),
    Document(page_content="RAG retrieves relevant context for an LLM.", metadata={"topic": "rag"}),
    Document(page_content="Agents can call tools to act.", metadata={"topic": "agents"}),
]

# The following assumes you already have a vector store and embeddings.
# vector_store = ...
# retriever = vector_store.as_retriever(search_kwargs={"k": 3})
# results = retriever.invoke("What is RAG?")

### MMR idea

In [ ]:
# MMR balances relevance and diversity.
# The usual goal is:
# selected = relevant AND not redundant
#
# In LangChain vector-store retrievers, MMR is commonly configured with:
#
# vector_store.as_retriever(
#     search_type="mmr",
#     search_kwargs={"k": 4, "fetch_k": 12, "lambda_mult": 0.5}
# )

### Simple keyword baseline

In [ ]:
import re

def keyword_search(query, docs, k=3):
    terms = set(re.findall(r"\w+", query.lower()))
    scored = []
    for d in docs:
        words = set(re.findall(r"\w+", d.page_content.lower()))
        score = len(terms & words)
        scored.append((score, d))
    scored.sort(key=lambda x: x[0], reverse=True)
    return scored[:k]

for score, doc in keyword_search("RAG context", docs):
    print(score, doc.page_content)

### Hybrid idea

In [ ]:
# A simple hybrid strategy:
# semantic_score and keyword_score are normalized to [0, 1]
# final_score = alpha * semantic_score + (1 - alpha) * keyword_score
#
# Tune alpha on a labeled retrieval set instead of guessing.

## Evaluation

Track Recall@K, Precision@K, MRR, and Hit Rate on labeled questions. Compare semantic-only, keyword-only, and hybrid retrieval.

## Improvement ideas

Add query rewriting, metadata filters, MMR, reranking, hybrid retrieval, and adaptive top-k. Use an evaluation set to choose settings rather than relying on intuition.

## Practice exercises

1. Build a 20-question retrieval benchmark.
2. Add MRR.
3. Tune hybrid alpha from 0.0 to 1.0.

## Final checklist

- [ ] I can explain the concept in simple words.
- [ ] I can describe where it fits in a GenAI system.
- [ ] I can run and modify the code.
- [ ] I know at least one way to evaluate it.
- [ ] I know what to change when it fails.

## Research / reference notes


- LangChain Python reference: https://reference.langchain.com/python/langchain/overview
- LangGraph Python reference: https://reference.langchain.com/python/langgraph/overview
- Ragas metrics: https://docs.ragas.io/en/latest/concepts/metrics/available_metrics/
- OpenAI function calling guide: https://developers.openai.com/api/docs/guides/function-calling
- OpenAI Responses API: https://developers.openai.com/api/reference/cli/resources/responses/methods/create